# Octavo

> Build nbdev documentation with Octavo
- order: 21

In [ ]:
#| default_exp octavo

Octavo is an alternative to Quarto for building a documentation website from nbdev notebooks. Install `nbdev[octavo]` to use it. A project selects Octavo by placing `octavo.yml` in its configured notebooks directory. Projects without that file continue to use Quarto and do not import Octavo.

The usual `nbdev-docs` command builds the site. Octavo currently builds all pages together. File-selection options are rejected unless left at their defaults; `n_workers` affects only Quarto builds.

Run `nbdev-export` before building to update the generated Python modules and symbol indexes. `nbdev-readme` and `nbdev-contributing` also use Octavo for these projects. `nbdev-prepare` still exports, tests, and cleans notebooks, but skips Quarto configuration generation.

For an existing Quarto project, `nbdev-migrate-octavo` converts documentation sources in place. The converted sources are not intended to work unchanged with both builders. Octavo support for `nbdev-preview` is not yet implemented.

## Migration finishing prompt

After running `nbdev-migrate-octavo`, use this prompt with an assistant that can inspect your project. `_quarto.yml` remains available for comparison. Migration changes sources in place; it does not keep them compatible with both renderers.

```markdown
Help finish this project's migration from Quarto to Octavo. Inspect the project before proposing changes. Agree on the changes with me before editing. Preserve unrelated work. Do not reset a checkout, clean untracked files, commit, or deploy.

Read the retained _quarto.yml, nbdev.yml, pyproject.toml, and generated octavo.yml. Inspect page metadata, stylesheets, render scripts, custom HTML, build hooks, and deployment workflows. Compare against the original site or screenshots where available. If no visual reference is available, say so rather than claiming visual parity.

Automatic migration renames sources and updates selected document references, directives, metadata, and site settings. It does not port the site's design. It does not copy format.html.css into octavo.yml. It leaves stylesheet files and page-layout metadata unchanged. Existing page-level CSS references can still load; inspect those too. An existing octavo.yml is preserved rather than regenerated.

Identify which of these checks apply, then propose a small migration plan:

- Styles and custom pages: inspect the rendered Octavo HTML before adapting selectors. Check homepage grids, buttons, spacing, responsive behavior, and dependencies on Bootstrap or Quarto classes. Reuse compatible CSS selectively. Do not enable an old stylesheet wholesale or copy a compatibility layer without checking the current DOM.
- Layouts and themes: review page-layout, sidebars, titles, TOCs, and theme settings. Choose Octavo settings explicitly. Do not assume that renaming a layout preserves its meaning. Octavo's layout: custom is an explicit choice for a custom page, not a general Quarto layout adapter. Check ordinary documentation pages as well as landing pages and blog listings.
- Embedded content: inspect iframes, videos, raw HTML, fenced divs, callouts, tabs, shortcodes, and format-conditional blocks used by this project. Where needed, adapt HTML-only content to raw {=html} blocks. Verify the rendered result and relevant Markdown outputs rather than rewriting every block by pattern.
- Render scripts and documentation: inspect renamed .md.py scripts and generated assets. Update tutorials and examples that still describe Quarto-specific commands, file extensions, or intermediate files. Source renaming alone does not make those explanations correct. Review executable scripts and custom processors before running a build.
- Navigation and identity: check navbar destinations, dropdowns, icons, favicon, footer, and existing public URLs. Restore omitted icons and the project's actual footer where needed. Do not infer license text from a project name.
- Additional build outputs: identify artifacts previously generated by Quarto hooks or custom scripts. If the project needs API lists or LLM context files, adapt its own generator using nbdev's docs_post_build callback. Do not add those artifacts to projects that do not need them. Check llms.txt destinations against actual output files, including old .html.md links and malformed Markdown links.
- Deployment: workflows are not migrated. Check documentation dependencies, replace direct Quarto build commands where needed, and verify the published directory. Octavo's output is relative to the notebooks directory. Use the ordinary nbdev-docs command. Do not copy a test harness's import-path workarounds or obsolete builder calls into production.

After approval, make the agreed changes in small steps. Export notebook-authored modules before building. Run nbdev-docs and the relevant repository Markdown commands. Inspect representative pages at desktop and narrow widths. Check navigation, anchors, images, embedded content, generated assets, and any additional build artifacts. A successful build alone does not establish a successful migration.

Report the changes, checks actually run, unresolved differences, and decisions requiring my review. Keep the original configuration until the migration has been reviewed. Do not copy test-only galleries or fixtures into the site unless I request them.
```

In [ ]:
#| export
from contextlib import redirect_stdout
from io import StringIO
from fastcore.utils import *
from fastcore.script import call_parse
import yaml
from octavo import *
from mdhtml import md2gfm
from nbdev.config import get_config, import_obj, _load_toml, read_nb
from nbdev.doclinks import NbdevLookup
from nbdev.process import NBProcessor
from nbdev.processors import FilterDefaults, add_links
from nbdev.serve import _is_qpy
from nbdev.sync import write_nb
from nbdev.quarto import _doc_mtime_not_older

In [ ]:
#| hide
import tempfile
from copy import deepcopy
from fastcore.test import *
from fastcore.nbio import new_nb, mk_cell, write_nb
from nbdev.quarto import nbdev_docs

## Notebook processing

Notebooks still need nbdev’s documentation processing before Octavo renders them. `OctavoProc` reuses that processing, including the project’s `doc_procs` and `skip_procs` settings.

It keeps directives such as `#| echo: false` for Octavo to interpret. It also places generated page metadata, such as the title, in a Markdown cell that Octavo can read. The caller chooses whether generated symbol links point into the current site or to published documentation. Links written explicitly by the author are unchanged.

In [ ]:
#| export
class OctavoProc(FilterDefaults):
    "Prepare notebook content, metadata, and symbol links for Octavo"
    def __init__(self, local_lib=None, strip_libs=None): self.nl = NbdevLookup(strip_libs=strip_libs or local_lib, local_lib=local_lib)
    def add_links(self, cell): return add_links(cell, nl=self.nl)
    def procs(self): return L(self.add_links if o is add_links else o for o in super().procs())
    def nb_proc(self, nb): return NBProcessor(nb=nb, procs=self.procs(), rm_directives=False)

    def __call__(self, nb):
        super().__call__(nb)
        if getattr(nb, 'frontmatter_', None) and nb.cells[0].cell_type=='raw': nb.cells[0].cell_type = 'markdown'

In [ ]:
source = new_nb([mk_cell('# Example\n\n> A notebook with rendering directives', 'markdown'),
                 mk_cell('Use `ModuleMaker`. See [production docs](https://nbdev.fast.ai/api/maker.html#modulemaker).', 'markdown'),
                 mk_cell('#| echo: false\n1 + 1')])
source.path_ = Path('example.ipynb')
nb = deepcopy(source)
OctavoProc(local_lib='nbdev')(nb)
test_eq(nb.cells[0].cell_type, 'markdown')
text = '\n'.join(c.source for c in nb.cells)
assert '[`ModuleMaker`](/api/maker.html#modulemaker)' in text
assert '[production docs](https://nbdev.fast.ai/api/maker.html#modulemaker)' in text
assert '#| echo: false' in text
quarto_nb = deepcopy(source)
FilterDefaults()(quarto_nb)
test_eq(quarto_nb.cells[0].cell_type, 'raw')
assert '#| echo: false' in '\n'.join(c.source for c in quarto_nb.cells)
nb.cells[0].source

'---\ndescription: A notebook with rendering directives\noutput-file: example.html\ntitle: Example\n\n---\n\n'

A website build needs to distinguish this project’s API from dependencies’ APIs. References to this project should lead to pages in the site being built; references to dependencies should lead to their published documentation.

nbdev identifies the project’s symbol indexes through `[project.entry-points.nbdev]` in `pyproject.toml`. It cannot infer those names from the package’s installation name. In the example below, the distribution is named `different-distribution`, but its symbol index is registered as `nbdev`.

In [ ]:
#| export
def local_libraries(cfg):
    "Names of the project's declared nbdev symbol indexes"
    proj = _load_toml(cfg.config_file).get('project', {})
    return tuple(proj.get('entry-points', {}).get('nbdev', {}))

In [ ]:
with tempfile.TemporaryDirectory() as d:
    project = Path(d)
    (project/'pyproject.toml').write_text('[project]\nname = "different-distribution"\n[project.entry-points.nbdev]\nnbdev = "nbdev._modidx:d"\n[tool.nbdev]\n')
    libs = local_libraries(get_config(project))
    test_eq(libs, ('nbdev',))
    lookup = NbdevLookup(local_lib=libs)
    assert lookup.doc('nbdev.maker.ModuleMaker').startswith('/api/maker.html#')
    assert NbdevLookup().doc('nbdev.maker.ModuleMaker').startswith('https://')
    nb = deepcopy(source)
    with working_directory(project): OctavoProc(local_lib=libs)(nb)
    assert '[`ModuleMaker`](/api/maker.html#modulemaker)' in '\n'.join(c.source for c in nb.cells)

## Render scripts

Render scripts generate documentation content during a build. Their filenames specify what they produce: `page.md.py` prints Markdown for a page, while `drawing.svg.py` prints an SVG image.

nbdev executes each script and captures its printed output. Generated Markdown goes through Octavo’s page renderer. Other output is written verbatim under the filename without `.py`; for example, `drawing.svg.py` produces `drawing.svg`.

A `.md.py` script can put YAML page metadata in its opening docstring. nbdev prepends that metadata to the printed Markdown.

Render-script source files are not published. Ordinary single-suffix `.py` files are copied as assets rather than executed. An exception in a render script stops the build.

In [ ]:
#| export
def render_script(path):
    "Execute a double-suffix render script and return its printed output with optional page metadata"
    path = Path(path)
    if path.suffix!='.py' or len(path.suffixes)<2: return
    fm = _is_qpy(path) if path.name.endswith('.md.py') else None
    f = StringIO()
    with redirect_stdout(f): exec(compile(path.read_text(encoding='utf-8'), str(path), 'exec'), {})
    return (f'---\n{fm}\n---\n\n' if fm else '') + f.getvalue()

In [ ]:
#| export
def script_pages(root):
    "Render scripts under `root`: `*.md.py` as `Page`s, anything else as `GeneratedFile`s"
    root,pages,files = Path(root),L(),L()
    for o in sorted(root.rglob('*.py')):
        rel = o.relative_to(root)
        if hidden(rel) or (txt:=render_script(o)) is None: continue
        dest = rel.with_suffix('')
        if dest.suffix=='.md': pages.append(Page(str(rel), txt, nm=page_nm(dest)))
        else: files.append(GeneratedFile(str(rel), str(dest), txt))
    return pages,files

In [ ]:
with tempfile.TemporaryDirectory() as d:
    d = Path(d)
    (d/'01_intro.md.py').write_text('"""---\ntitle: Hi\n---"""\nprint("# Generated")\nprint("body")')
    (d/'images').mkdir(); (d/'images/dot.svg.py').write_text('print("<svg/>")')
    (d/'plain.md.py').write_text('print("# No frontmatter")')
    (d/'ordinary.py').write_text('x = 1')
    (d/'_skip').mkdir(); (d/'_skip/no.md.py').write_text('"""---\n---"""\nprint("no")')
    pages,files = script_pages(d)
    test_eq([p.dest for p in pages], ['intro.html','plain.html'])
    test_eq(pages[0].text, '---\ntitle: Hi\n---\n\n# Generated\nbody\n')
    test_eq(pages[0].meta['title'], 'Hi')
    test_eq(pages[1].text, '# No frontmatter\n')
    test_eq([(f.src,f.dest,f.content) for f in files], [('images/dot.svg.py','images/dot.svg','<svg/>\n')])
    test_is(render_script(d/'ordinary.py'), None)
    (d/'bad.md.py').write_text('1/0')
    with expect_fail(ZeroDivisionError, contains='division'): script_pages(d)

## Post-build callback

Some projects publish extra files alongside their documentation, such as an API list or an LLM context file. Use a post-build callback to generate these after the website has been built.

Set `docs_post_build` to an importable `module:function` in `pyproject.toml`:

```toml
[tool.nbdev]
docs_post_build = "my_project.docbuild:build_extras"
```

After a successful site build, nbdev calls the function once with `(out, cfg)`. `out` is the absolute directory containing the completed site. `cfg` is the nbdev project configuration. The callback writes additional files into `out`; its return value is ignored. Callback errors fail the build. Without this setting, no callback runs.

Author the callback in a notebook and export it before building the docs, as with custom processors. Its dependencies belong to the project's documentation setup.

## Build configuration

`octavo.yml` contains named settings, such as `title: Demo`. An empty file selects Octavo’s defaults. A YAML list or a single value such as `false` cannot supply those settings.

The build checks this file before running render scripts or notebook processors. Invalid configuration therefore fails before project code executes.

In [ ]:
#| export
def octavo_config(root):
    "Read `octavo.yml` as a settings dictionary; return an empty dictionary for an empty file"
    root = Path(root)
    if not (root/'octavo.yml').is_file(): raise FileNotFoundError(root/'octavo.yml')
    settings = yaml.safe_load((root/'octavo.yml').read_text())
    if settings is None: return {}
    if not isinstance(settings, dict): raise ValueError(f'{root / "octavo.yml"} must contain a YAML mapping')
    return settings

In [ ]:
with tempfile.TemporaryDirectory() as d:
    root = Path(d)
    test_fail(lambda: octavo_config(root), contains='octavo.yml')
    for text in ('- not a mapping', '[]', 'false', '0'):
        (root/'octavo.yml').write_text(text)
        test_fail(lambda: octavo_config(root), contains='mapping')
    (root/'octavo.yml').write_text('')
    test_eq(octavo_config(root), {})

Links to this project’s API should stay within the website being built. For example, when building nbdev’s website, a generated link for `ModuleMaker` points to `/api/maker.html#modulemaker` rather than the published nbdev site.

Links to dependencies still point to their published documentation. An absolute URL written by the author is not changed.

In [ ]:
#| export
def octavo_docs(
    path:str=None, # Path to the project or its notebooks
):
    "Create Octavo docs and run the optional docs_post_build callback"
    cfg = get_config(path)
    settings = {'output': cfg.doc_path, **octavo_config(cfg.nbs_path)}
    with working_directory(cfg.config_path):
        pages,files = script_pages(cfg.nbs_path)
        docs = scan(cfg.nbs_path, nbproc=OctavoProc(local_lib=local_libraries(cfg))) + pages
        site = Site(docs, root=cfg.nbs_path, files=files, **settings)
        if (miss := site.build()): print(miss)
        if (post := cfg.get('docs_post_build')):
            out = Path(site.settings['output']).expanduser()
            if not out.is_absolute(): out = cfg.nbs_path/out
            import_obj(post)(out.resolve(), cfg)

## Build a documentation site

`nbdev-docs` reads `octavo.yml` and processes notebooks and render scripts together. It does not migrate sources or change `_quarto.yml`. This temporary project builds linked pages and generated assets, then demonstrates that invalid configuration is rejected before executing a render script.

In [ ]:
build_tmp = tempfile.TemporaryDirectory()
project = Path(build_tmp.name)
nbs = project/'nbs'
nbs.mkdir()
(project/'pyproject.toml').write_text('[project]\nname = "demo"\n[tool.nbdev]\nnbs_path = "nbs"\nskip_procs = "insert_warning"\n')
(nbs/'octavo.yml').write_text('title: Demo\noutput: ../_octavo\nsite_url: https://example.com\n')
(nbs/'_quarto.yml').write_text('project:\n  type: website\n')
(nbs/'index.md').write_text('# Home\n\nRead the [guide](guide.ipynb).')
guide = new_nb([mk_cell('# Guide\n\n> A notebook page', 'markdown'), mk_cell('Notebook content.', 'markdown')])
write_nb(guide, nbs/'guide.ipynb')
(nbs/'generated.md.py').write_text('print("# Generated page")')
(nbs/'image.svg.py').write_text('print("<svg/>")')

15

The ordinary build command leaves Quarto configuration unchanged. Without `output` in `octavo.yml`, it uses nbdev’s project-relative `doc_path`. An explicit Octavo output takes precedence and is relative to the notebooks directory. Here, `../_octavo` overrides the default project-level `_docs` directory.

In [ ]:
with working_directory(project): run('nbdev-docs')
out = project/'_octavo'
assert 'guide.html' in (out/'index.html').read_text()
assert 'Notebook content.' in (out/'guide.html').read_text()
assert 'AUTOGENERATED' not in (out/'guide.md').read_text()
assert (out/'generated.html').exists()
test_eq((out/'image.svg').read_text(), '<svg/>\n')
test_eq((nbs/'_quarto.yml').read_text(), 'project:\n  type: website\n')

Existing scripts can pass the default file-selection options explicitly. Octavo accepts those calls, but still builds the whole site. For normal use, run `nbdev-docs` without file-selection options.

In [ ]:
with working_directory(project): run(r'nbdev-docs --file-re "\.(?:ipynb|qmd|html)$"')
nbdev_docs(project, file_glob=None)

The Octavo integration currently builds the whole site. Non-default file filters are rejected because selective builds are not implemented.

Invalid configuration must fail before any render script runs.

In [ ]:
test_fail(lambda: nbdev_docs(project, file_glob='*.ipynb'), contains='file_glob')
(nbs/'octavo.yml').write_text('- invalid')
(nbs/'generated.md.py').write_text('raise RuntimeError("script ran before validation")')
test_fail(lambda: nbdev_docs(project), contains='mapping')
build_tmp.cleanup()

## Repository Markdown

`nbdev-readme` and `nbdev-contributing` render the configured notebooks to the project root without Quarto. They use stored outputs and do not execute ordinary code cells. nbdev processors still run, including custom processors and generated API documentation. Titles and descriptions become Markdown. Symbol links use published documentation URLs.

Images stored in notebook outputs are extracted into a directory beside the generated Markdown. The directory uses the source notebook’s name: `index.ipynb` produces `index_files`, even though the generated document is named `README.md`. This matches nbdev’s Quarto convention.

Authored relative links are copied unchanged; their destinations are not adjusted for the document’s new location.

A missing source notebook is skipped. With `chk_time=True`, an existing document is skipped when its modification time is at least as recent as the source notebook’s.

In [ ]:
#| export
def octavo_readme(
    path:str=None, # Path to the project or its notebooks
    chk_time:bool=False, # Only build if the source is newer
    contributing:bool=False, # Render CONTRIBUTING instead of README
):
    "Render a repository Markdown document from a notebook's stored outputs"
    cfg = get_config(path)
    nbname = cfg.get('contributing_nb', 'contributing.ipynb') if contributing else cfg.readme_nb
    root = Path(path).absolute() if path else cfg.nbs_path
    src = root/nbname
    if not src.exists(): src = cfg.nbs_path/nbname
    if not src.exists(): return
    name = 'CONTRIBUTING' if contributing else 'README'
    dest = cfg.config_path/f'{name}.md'
    if chk_time and _doc_mtime_not_older(dest, src): return
    octavo_config(cfg.nbs_path)
    with working_directory(cfg.config_path):
        proc = OctavoProc(strip_libs=local_libraries(cfg))
        doc = md2doc(ipynb2md(src, nbproc=proc))
        head = f'# {doc.meta["title"]}\n\n' if doc.meta.get('title') else ''
        if desc:=doc.meta.get('description'): head += f'{desc}\n\n'
        md2gfm(head + doc.body, dest=dest, imgdir=cfg.config_path/f'{src.stem}_files', raw=('md', 'html'))

In [ ]:
readme_tmp = tempfile.TemporaryDirectory()
project = Path(readme_tmp.name)
nbs = project/'nbs'
nbs.mkdir()
(project/'pyproject.toml').write_text(r'''[project]
name = "different-distribution"
[project.entry-points.nbdev]
nbdev = "nbdev._modidx:d"
[tool.nbdev]
nbs_path = "nbs"
contributing_nb = "help.ipynb"
''')
(nbs/'octavo.yml').write_text('{}')

2

README generation should use a notebook’s saved results without rerunning its ordinary code cells. The example below constructs a notebook containing a saved image and code that raises an exception if executed.

Successful rendering must include the saved image without running that code. This matches Quarto’s `--no-execute` behaviour.

In [ ]:
png = 'iVBORw0KGgoAAAANSUhEUgAAAAEAAAABCAQAAAC1HAwCAAAAC0lEQVR42mP8/x8AAwMCAO+aX1sAAAAASUVORK5CYII='
image = mk_cell('#| echo: false\nraise RuntimeError("must not execute")')
image.outputs = [dict(output_type='display_data', metadata={}, data={'image/png': png})]
book = new_nb([mk_cell('# Example\n\n> Repository document', 'markdown'),
               mk_cell('Use `ModuleMaker`. [Guide](guide.md). [Reference][ref]\n\n[ref]: guide.md', 'markdown'),
               image, mk_cell('#| include: false\n"excluded cell"')])
write_nb(book, nbs/'index.ipynb')
write_nb(book, nbs/'help.ipynb')

In [ ]:
book

```python
{ 'cells': [ { 'cell_type': 'markdown',
               'directives_': {},
               'id': '03d65b6e',
               'idx_': 0,
               'lang_': 'python',
               'metadata': {},
               'source': '# Example\n\n> Repository document'},
             { 'cell_type': 'markdown',
               'directives_': {},
               'id': 'bcf12a9e',
               'idx_': 1,
               'lang_': 'python',
               'metadata': {},
               'source': 'Use `ModuleMaker`. [Guide](guide.md). '
                         '[Reference][ref]\n'
                         '\n'
                         '[ref]: guide.md'},
             { 'cell_type': 'code',
               'directives_': {},
               'execution_count': None,
               'id': '0b2fc6cb',
               'idx_': 2,
               'lang_': 'python',
               'metadata': {},
               'outputs': [ { 'data': { 'image/png': 'iVBORw0KGgoAAAANSUhEUgAAAAEAAAABCAQAAAC1HAwCAAAAC0lEQVR42mP8/x8AAwMCAO+aX1sAAAAASUVORK5CYII='},
                              'metadata': {},
                              'output_type': 'display_data'}],
               'source': '#| echo: false\n'
                         'raise RuntimeError("must not execute")'},
             { 'cell_type': 'code',
               'directives_': {},
               'execution_count': None,
               'id': 'ba6db483',
               'idx_': 3,
               'lang_': 'python',
               'metadata': {},
               'outputs': [],
               'source': '#| include: false\n"excluded cell"'}],
  'metadata': {},
  'nbformat': 4,
  'nbformat_minor': 5}
```

README and CONTRIBUTING use separate source notebooks and produce separate Markdown files. This test gives both notebooks identical content to check that both commands handle hidden code, saved images, and links consistently.

In [ ]:
cwd = Path.cwd()
octavo_readme(project)
with working_directory(project):
    run('nbdev-readme')
    run('nbdev-contributing')

A README is read in the repository, not alongside the generated API pages. Its generated symbol links therefore point to published documentation—even when they refer to this project’s own API.

For example, nbdev’s README links `ModuleMaker` to the published nbdev website. The nbdev website itself uses a local API link. This difference depends on where the document will be read, not on which library the symbol belongs to.

In [ ]:
for name,stem in (('README', 'index'), ('CONTRIBUTING', 'help')):
    text = (project/f'{name}.md').read_text()
    assert '# Example' in text and 'Repository document' in text
    assert 'https://nbdev.fast.ai/api/maker.html#modulemaker' in text
    assert '(guide.md)' in text and '[ref]: guide.md' in text
    assert 'must not execute' not in text and 'excluded cell' not in text
    assert 'data:image' not in text and f'{stem}_files/' in text
    assert list((project/f'{stem}_files').glob('*.png'))

An up-to-date README should not be rewritten. Projects without a CONTRIBUTING source should still build their other documents.

In [ ]:
before = (project/'README.md').stat().st_mtime_ns
octavo_readme(project, chk_time=True)
test_eq((project/'README.md').stat().st_mtime_ns, before)
assert not (nbs/'_quarto.yml').exists()
(nbs/'help.ipynb').unlink()
octavo_readme(project, contributing=True)
readme_tmp.cleanup()

## Source migration

`nbdev-migrate-octavo` converts Quarto documentation sources in place. Commit or back up the project before running it. The command:

1. Renames `.qmd` documents to `.md` and `.qmd.py` render scripts to `.md.py`.
2. Rewrites references to renamed documents.
3. Replaces Quarto content-visible block markers with raw-format fences.
4. Converts selected page-metadata names, such as `subtitle` to `description`.
5. Creates `octavo.yml` from selected Quarto settings when no Octavo configuration exists.
6. Leaves `_quarto.yml` unchanged for reference.

The presence of `octavo.yml` selects Octavo.

Migration converts sources and configuration, not the site’s design. CSS, layouts, and themes need project-specific review. Stylesheet files and `page-layout` metadata remain unchanged. Quarto stylesheet references are not copied into `octavo.yml`.

In [ ]:
#| export
def _qmd_moves(path):
    "`.qmd` documents and `.qmd.py` render scripts under `path`, mapped to Octavo names"
    path = Path(path)
    ps = L(path.rglob('*.qmd')) + L(path.rglob('*.qmd.py'))
    return {p:p.with_name(re.sub(r'\.qmd(?=\.py$|$)', '.md', p.name)) for p in sorted(ps)
            if not any(o[0] in '._' for o in p.relative_to(path).parts)}

Renaming a document also requires updating references to it. A page beside `guide.qmd` refers to it differently from a page in a subdirectory: `guide.qmd` versus `../guide.qmd`. Migration calculates replacement paths from the documentation root and from each source file’s directory.

The current implementation performs text replacement rather than parsing links. Matching text outside link destinations can also change.

In [ ]:
#| export
def _qmd_repls(src, path, moves):
    "Old and new spellings of each moved document, as seen from `src`"
    def _rel(p, start): return os.path.relpath(p, start).replace(os.sep, '/')
    res = {}
    for old,new in moves.items():
        res[_rel(old, path)] = _rel(new, path)
        res[_rel(old, src.parent)] = _rel(new, src.parent)
    return sorted(res.items(), key=lambda o: len(o[0]), reverse=True)

def repl_qmd_links(txt, moves, src, root):
    "Rewrite relative and root-relative `.qmd` link targets to `.md`"
    for old,new in _qmd_repls(src, root, moves): txt = txt.replace(old, new)
    return txt

Quarto lets authors mark a block of content for a particular output format—for example, HTML for the website or Markdown for a README. Migration replaces Quarto’s block markers with mdhtml's raw-format fences.

For example, this Quarto block:

````markdown
::: {.content-visible when-format="html"}
<p>Only include this in HTML output.</p>
:::
````

becomes:

````markdown
```{=html}
<p>Only include this in HTML output.</p>
```
````

The function changes the surrounding markers, not the content inside them. Quarto’s `markdown`, `commonmark`, and `md` format names all become `md`. If an HTML block has a `style` attribute, the function preserves it by wrapping the content in a `<div>` with that style.

This conversion changes block markers, not the language of the body. Markdown inside an HTML-targeted block is not converted to HTML. The current helper also does not preserve `unless-format` conditions. Review these cases before relying on the converted output.

In [ ]:
#| export
_re_cv = re.compile(r'^:{3,}[^\S\r\n]*\{([^}]*\.content-visible[^}]*)\}[^\S\r\n]*\n(.*?)\n:{3,}[^\S\r\n]*$', re.MULTILINE | re.DOTALL)

def _repl_cv_match(m):
    attrs,body = m.group(1),m.group(2).strip()
    m_fmt = re.search(r'when-format=["\']?(\w+)["\']?', attrs)
    fmt = m_fmt.group(1).lower() if m_fmt else 'html'
    target = 'md' if fmt in ('markdown', 'commonmark', 'md') else fmt
    if target=='html':
        if (m_style := re.search(r'style=["\']([^"\']+)["\']', attrs)):
            body = f'<div style="{m_style.group(1)}">\n{body}\n</div>'
    return f'```{{={target}}}\n{body}\n```'

def repl_content_visible(s):
    "Convert Quarto `::: {.content-visible when-format=...}` to ````{=md}```` / ````{=html}````"
    return _re_cv.sub(_repl_cv_match, s)

Explicit Octavo keys take precedence over legacy aliases. Only top-level keys are normalized; nested metadata retains its original meaning. `page-layout` remains unchanged for manual review because Quarto and Octavo layouts are not interchangeable.

In [ ]:
#| export
def _dump_fm(fm): return yaml.dump(fm, sort_keys=False, allow_unicode=True, width=10**9)

def repl_frontmatter_keys(s):
    "Normalize legacy and Quarto frontmatter keys for Octavo"
    fm,body = frontmatter(s)
    if not fm: return s
    changed = False
    aliases = [('categories','tags'), ('subtitle','description'), ('summary','description')]
    for old,new in aliases + [(k,k.replace('-','_')) for k in fm if '-' in k and k!='page-layout']:
        if old not in fm: continue
        fm.setdefault(new, fm.pop(old))
        changed = True
    if fm.pop('toc', None) is False:
        fm['toc_depth'] = 0
        changed = True
    if changed: return f'---\n{_dump_fm(fm)}---\n\n{body.lstrip(chr(10))}'
    return s

Render scripts keep their executable body. Only the leading YAML docstring needs the same metadata conversion as a document.

In [ ]:
#| export
_script_fm = re.compile(r'\A(?P<q>"{3}|\'{3})---\n(?P<fm>.*?)\n---(?P=q)', re.DOTALL)

def repl_script_frontmatter_keys(s):
    "Normalize a render script's leading YAML docstring for Octavo"
    if not (m := _script_fm.match(s)): return s
    old = f"---\n{m.group('fm')}\n---\n"
    new = repl_frontmatter_keys(old)
    if new == old: return s
    fm,_ = frontmatter(new)
    return f"{m.group('q')}---\n{_dump_fm(fm)}---{m.group('q')}" + s[m.end():]

Documentation conversion must not rewrite executable notebook code or discard saved results. `migrate_nb_cells` applies the text conversions only to Markdown cells. Code cells and their saved outputs are retained.

`migrate_nb_cells` and `migrate_md_file` write a file only when a conversion changes its content.

In [ ]:
#| export
def _apply_procs(txt, procs):
    for p in procs: txt = p(txt)
    return txt

def migrate_nb_cells(path, procs):
    "Apply `procs` to all markdown cells in a notebook in-place"
    nb,changed = read_nb(path),False
    for c in nb.cells:
        if c.cell_type != 'markdown' or not c.get('source'): continue
        new_src = _apply_procs(c.source, procs)
        if new_src != c.source: c.source,changed = new_src,True
    if changed: write_nb(nb, path)

def migrate_md_file(path, procs):
    "Apply `procs` to a Markdown/QMD file in-place"
    txt = path.read_text()
    new_txt = _apply_procs(txt, procs)
    if new_txt!=txt: path.write_text(new_txt)

In [ ]:
#| export
def _nav_href(h):
    if not h or '://' in h or h.startswith(('#','mailto:')): return h
    return re.sub(r'\.(?:ipynb|qmd|md)(?=[?#]|$)', '.html', h)

def _nav_item(item):
    res = {'text': item['text']}
    if 'menu' in item: res['children'] = [{'text': m['text'], 'href': _nav_href(m['href'])} for m in item['menu'] if 'href' in m]
    elif 'href' in item: res['href'] = _nav_href(item['href'])
    return res

Icon names belong to the renderer. Unsupported icons are reported rather than emitted as broken tool links.

In [ ]:
#| export
_ICON_MAP = {'twitter':'x', 'twitter-x':'x'}
_ICON_LABELS = {'github':'GitHub', 'rss':'RSS'}

def _tool_item(item):
    "An octavo `tools` entry from a Quarto icon link, or None where octavo has no such icon"
    from octavo.theme import ICONS
    ico = _ICON_MAP.get(item['icon'], item['icon'])
    if ico not in ICONS: return None
    return {'icon': ico, 'href': _nav_href(item['href']), 'label': item.get('aria-label') or item.get('text') or _ICON_LABELS.get(ico, ico.title())}

Configuration is parsed before any source files are changed. Empty YAML uses defaults; a scalar or list cannot represent site settings.

In [ ]:
#| export
def _quarto_config(nbs_path):
    qy_p,ny_p = nbs_path/'_quarto.yml',nbs_path/'nbdev.yml'
    qy = yaml.safe_load(qy_p.read_text()) if qy_p.exists() else {}
    ny = yaml.safe_load(ny_p.read_text()) if ny_p.exists() else {}
    for fname,settings in ((qy_p,qy), (ny_p,ny)):
        if settings is not None and not isinstance(settings, dict): raise ValueError(f'{fname} must contain a YAML mapping')
    return qy or {},ny or {}

nbdev already knows where to put the website through `doc_path`, relative to the project root. Migration leaves `output` out of `octavo.yml` when that destination is sufficient.

An explicit Quarto `output-dir` can name a different destination. Migration preserves that override, expressing it relative to the notebooks directory for Octavo. For example, a project-level `website` directory becomes `../website` when notebooks live in `nbs`.

In [ ]:
#| export
def _octavo_output(nbs_path, qy, ny):
    "Return a Quarto output override relative to notebooks, or None when doc_path suffices"
    out_dir = (ny.get('project', {}) or {}).get('output-dir') or (qy.get('project', {}) or {}).get('output-dir')
    if not out_dir: return
    cfg = get_config(nbs_path)
    out_dir = (cfg.config_path/Path(out_dir).expanduser()).resolve()
    if out_dir == cfg.doc_path.resolve(): return
    return os.path.relpath(out_dir, Path(nbs_path).resolve()).replace(os.sep, '/')

Quarto and Octavo use different settings for navigation bars. Migration converts text links and dropdowns into Octavo navigation links, and icon links into tools. Links to documentation sources are changed to HTML destinations so visitors reach rendered pages. Explicit accessibility labels are preserved. Icons not bundled with Octavo are reported and omitted for manual replacement.

In [ ]:
#| export
def _octavo_nav(nbs_path, qy, ny):
    "Convert Quarto navbar settings into Octavo navigation links and icon tools"
    cfg = {}
    def _nav(k): return nested_idx(qy,'website','navbar',k) or nested_idx(ny,'website','navbar',k) or []
    if nav_left := _nav('left'): cfg['nav_links'] = [_nav_item(o) for o in nav_left if 'text' in o and ('href' in o or 'menu' in o)]
    links = L(_nav('right')+_nav('tools')).filter(lambda o: 'icon' in o and 'href' in o)
    tools = links.map(_tool_item)
    for o,t in zip(links,tools):
        if t is None: print(f"{nbs_path/'octavo.yml'}: dropped tool icon {o['icon']!r} ({o['href']}): not bundled by octavo. Add it back with `icon: <path-or-url>` to your own SVG.")
    if (tools := tools.filter()): cfg['tools'] = list(tools)
    return cfg

A migrated site needs its title, navigation, and other supported settings in Octavo’s format. `_create_octavo_cfg` collects those settings into a dictionary. An output override is included only when the Quarto destination differs from nbdev’s `doc_path`. The migration command writes the dictionary to `octavo.yml` only when that file does not already exist.

This is not a complete translation of Quarto configuration. Settings without a conversion still need manual review.

In [ ]:
#| export
def _create_octavo_cfg(nbs_path):
    "Return Octavo settings derived from `_quarto.yml` and `nbdev.yml`"
    nbs_path = Path(nbs_path).resolve()
    qy,ny = _quarto_config(nbs_path)
    ws_q,ws_n = qy.get('website', {}) or {},ny.get('website', {}) or {}
    fmt_html = (qy.get('format', {}) or {}).get('html', {}) or {}
    cfg = {}
    if (output := _octavo_output(nbs_path, qy, ny)) is not None: cfg['output'] = output
    title = ws_n.get('title') or ws_q.get('title')
    if title: cfg['title'] = title
    site_url = ws_n.get('site-url') or ws_q.get('site-url')
    if site_url: cfg['site_url'] = site_url.rstrip('/')
    if 'toc-depth' in fmt_html:
        try: cfg['toc_depth'] = max(1, int(fmt_html['toc-depth']) - 1)
        except (ValueError, TypeError): pass
    else: cfg['toc_depth'] = 0 if fmt_html.get('toc') is False else 3
    favicon = ws_n.get('favicon') or ws_q.get('favicon')
    if favicon: cfg['favicon'] = favicon
    cfg.update(_octavo_nav(nbs_path, qy, ny))
    return cfg

Switching builders requires source changes as well as new configuration: renamed documents need updated links, and Quarto-specific markup needs conversion. Migration checks rename conflicts and configuration before editing, preserves an existing `octavo.yml`, and retains `_quarto.yml` for comparison. Changes are made in place without automatic rollback, so commit or back up the project first.

In [ ]:
#| export
def _migrate_octavo(path):
    "Migrate documentation sources in place, create missing Octavo configuration, and return the rename mapping"
    path = Path(path).expanduser()
    if not path.is_dir(): raise NotADirectoryError(path)
    moves = _qmd_moves(path)
    for new in moves.values():
        if new.exists(): raise FileExistsError(f'Cannot rename to existing file: {new}')
    qy = path/'_quarto.yml'
    if (path/'octavo.yml').exists(): octavo_config(path)
    elif not qy.is_file(): raise FileNotFoundError(qy)
    cfg = _create_octavo_cfg(path) if qy.exists() and not (path/'octavo.yml').exists() else None

    srcs = [p for p in sorted(path.rglob('*')) if p.suffix in ('.ipynb','.qmd','.md')
            and not any(o[0] in '._' for o in p.relative_to(path).parts)]
    scripts = [p for p in sorted(path.rglob('*.qmd.py'))
               if not any(o[0] in '._' for o in p.relative_to(path).parts)]
    for src in srcs:
        procs = [partial(repl_qmd_links, moves=moves, src=src, root=path), repl_content_visible, repl_frontmatter_keys]
        if src.suffix=='.ipynb': migrate_nb_cells(src, procs)
        else:                    migrate_md_file(src, procs)
    for src in scripts: migrate_md_file(src, [repl_script_frontmatter_keys])

    for old,new in moves.items(): old.rename(new)

    if cfg: (path/'octavo.yml').write_text(yaml.dump(cfg, sort_keys=False))
    return moves

The default comes from the project configuration, so the command also works when notebooks live outside `nbs`.

In [ ]:
#| export
@call_parse
def nbdev_migrate_octavo(
    path:str=None, # Documentation directory (defaults to configured nbs_path)
):
    "Migrate Quarto-era nbdev project to Octavo"
    moves = _migrate_octavo(get_config().nbs_path if path is None else path)
    for old,new in moves.items(): print(f'{old} -> {new}')

A documentation project can mix written pages, notebooks, and pages generated by Python scripts. The following project includes all three. After migration, we inspect their filenames, metadata, links, and retained content.

In [ ]:
migration_tmp = tempfile.TemporaryDirectory()
d = migration_tmp.name
nbs = Path(d)/'nbs'
(Path(d)/'pyproject.toml').write_text('[project]\nname = "demo"\n[tool.nbdev]\nnbs_path = "nbs"\n')
(nbs/'blog/posts').mkdir(parents=True)
(nbs/'images').mkdir()
(nbs/'guide.qmd').write_text(r'''---
subtitle: A handy guide
toc-depth: 2
custom-key: keep-this
---

See [blog](blog/index.qmd). The `.qmd` format is mentioned here.

::: {.content-visible when-format="html" style="text-align: center"}
<iframe src="video"></iframe>
:::
''')
(nbs/'blog/index.qmd').write_text(r'''---
title: Blog
subtitle: All posts
sidebar: false
listing:
  categories: true
---

Back to [guide](../guide.qmd).
''')

115

Render scripts carry metadata in a Python docstring. Their program body and non-Markdown generated assets must remain unchanged.

In [ ]:
(nbs/'index.qmd.py').write_text(r'''"""---
title: Home
description: Write, test, document, and distribute software packages and technical articles — all in one place, your notebook.
page-layout: custom
toc: false
---"""
print("# Home")
''')
(nbs/'blog/posts/render.qmd.py').write_text('"""---\ntitle: Rendered\ntoc-depth: 3\ntoc_depth: 1\n---"""\nprint("# Rendered")\n')
(nbs/'images/circles.svg.py').write_text('"""---\n---"""\nprint("<svg/>")\n')

30

This project includes both supported and unsupported navbar icons. Migration should report the unsupported entry without losing the supported links.

In [ ]:
(nbs/'_quarto.yml').write_text(r'''website:
  title: demo
  favicon: favicon.png
  navbar:
    left:
      - text: Guide
        href: guide.qmd
    right:
      - icon: github
        href: https://github.com/x/y
      - icon: twitter
        href: https://twitter.com/x
        aria-label: Follow us
      - icon: bug
        href: https://x.y/issues
format:
  html:
    css: styles.css
    toc-depth: 4
''')
(nbs/'styles.css').write_text('body { color: red; }')

371

Notebook frontmatter and format-specific Markdown need the same conversion as standalone documents.

In [ ]:
nb = new_nb([mk_cell(r'''---
summary: Notebook summary
categories: [tutorial]
toc-depth: 2
---

[guide](guide.qmd) and [blog](blog/index.qmd)

::: {.content-visible when-format="markdown"}
![CI](badge.svg)
:::''', 'markdown')])
write_nb(nb, nbs/'index.ipynb')

Renaming includes nested documents and Markdown render scripts. Other generated asset types retain their filenames and content.

In [ ]:
quarto_before = (nbs/'_quarto.yml').read_bytes()
moves = _migrate_octavo(nbs)
test_eq({str(k.relative_to(nbs)):str(v.relative_to(nbs)) for k,v in moves.items()}, {
    'blog/index.qmd':'blog/index.md', 'blog/posts/render.qmd.py':'blog/posts/render.md.py',
    'guide.qmd':'guide.md', 'index.qmd.py':'index.md.py'})
assert not (nbs/'guide.qmd').exists() and (nbs/'guide.md').exists()
assert not (nbs/'blog/index.qmd').exists() and (nbs/'blog/index.md').exists()
assert not (nbs/'index.qmd.py').exists() and (nbs/'index.md.py').exists()
assert not (nbs/'blog/posts/render.qmd.py').exists() and (nbs/'blog/posts/render.md.py').exists()
test_eq((nbs/'images/circles.svg.py').read_text(), '"""---\n---"""\nprint("<svg/>")\n')

/private/var/folders/0h/5ktbjhg17ns6j8zgdsgvzvy80000gn/T/tmpcyu0zsjx/nbs/octavo.yml: dropped tool icon 'bug' (https://x.y/issues): not bundled by octavo. Add it back with `icon: <path-or-url>` to your own SVG.


Existing underscore keys take precedence in script metadata. `page-layout` is retained without creating `layout` or `page_layout`. Long descriptions must survive YAML serialization without inserted line continuations.

In [ ]:
idx_txt = (nbs/'index.md.py').read_text()
idx_fm = yaml.safe_load(_script_fm.match(idx_txt).group('fm'))
test_eq(idx_fm['page-layout'], 'custom')
test_eq(idx_fm['toc_depth'], 0)
assert 'layout' not in idx_fm and 'page_layout' not in idx_fm and 'toc' not in idx_fm
assert 'technical\\\n' not in idx_txt and '— all in one place' in idx_txt
render_txt = (nbs/'blog/posts/render.md.py').read_text()
render_fm = yaml.safe_load(_script_fm.match(render_txt).group('fm'))
test_eq(render_fm['toc_depth'], 1)
assert 'toc-depth' not in render_fm

The converted configuration retains the site identity and supported navigation. Quarto CSS is not enabled in `octavo.yml`. The original configuration and stylesheet remain unchanged for manual adaptation.

In [ ]:
test_eq((nbs/'_quarto.yml').read_bytes(), quarto_before)
assert (nbs/'octavo.yml').exists()
cfg = yaml.safe_load((nbs/'octavo.yml').read_text())
test_eq(cfg['title'], 'demo')
assert 'output' not in cfg
test_eq(cfg['toc_depth'], 3)
assert 'css' not in cfg
test_eq((nbs/'styles.css').read_text(), 'body { color: red; }')
test_eq(cfg['favicon'], 'favicon.png')
test_eq(cfg['nav_links'], [{'text': 'Guide', 'href': 'guide.html'}])
test_eq(cfg['tools'], [{'icon': 'github', 'href': 'https://github.com/x/y', 'label': 'GitHub'},
                       {'icon': 'x', 'href': 'https://twitter.com/x', 'label': 'Follow us'}])

Frontmatter aliases change at the top level. Relative links still need to reach the renamed document from its own directory.

In [ ]:
guide_fm,guide_body = frontmatter((nbs/'guide.md').read_text())
test_eq(guide_fm['description'], 'A handy guide')
assert 'subtitle' not in guide_fm
test_eq(guide_fm['toc_depth'], 2)
test_eq(guide_fm['custom_key'], 'keep-this')
assert 'toc-depth' not in guide_fm and 'custom-key' not in guide_fm

blog_fm,blog_body = frontmatter((nbs/'blog/index.md').read_text())
test_eq(blog_fm['description'], 'All posts')
test_eq(blog_fm['sidebar'], False)
test_eq(_nav_item(dict(text='Help', menu=[dict(text='Guide', href='guide.qmd')])),
        dict(text='Help', children=[dict(text='Guide', href='guide.html')]))
test_eq(blog_body.strip(), 'Back to [guide](../guide.md).')

Notebook Markdown receives the same metadata conversion. A second migration must leave the converted metadata unchanged.

In [ ]:
nb_fm,nb_body = frontmatter(read_nb(nbs/'index.ipynb').cells[0].source)
test_eq(nb_fm['description'], 'Notebook summary')
test_eq(nb_fm['tags'], ['tutorial'])
assert 'summary' not in nb_fm and 'categories' not in nb_fm
test_eq(nb_fm['toc_depth'], 2)
assert 'toc-depth' not in nb_fm

test_eq(_migrate_octavo(nbs), {})
test_eq(frontmatter((nbs/'guide.md').read_text())[0], guide_fm)
migration_tmp.cleanup()

A destination collision must stop migration before an existing file can be overwritten.

In [ ]:
with tempfile.TemporaryDirectory() as d:
    p = Path(d)
    (p/'page.qmd.py').touch(); (p/'page.md.py').touch()
    with expect_fail(FileExistsError, contains='page.md.py'): _migrate_octavo(p)

Malformed YAML must fail before any source rename. This differs from a valid YAML value of the wrong type.

In [ ]:
with tempfile.TemporaryDirectory() as d:
    p = Path(d)
    (p/'guide.qmd').write_text('# Guide')
    (p/'_quarto.yml').write_text('website: [invalid')
    with expect_fail(yaml.YAMLError): _migrate_octavo(p)
    assert (p/'guide.qmd').exists() and not (p/'guide.md').exists()

This project keeps notebooks in `docs/notebooks` and sets `doc_path` to `public`. Migration omits redundant output settings; the build still writes to the project’s `public` directory. A different Quarto destination is preserved.

In [ ]:
nested_tmp = tempfile.TemporaryDirectory()
d = nested_tmp.name
project = Path(d)
nbs = project/'docs/notebooks'
nbs.mkdir(parents=True)
(project/'pyproject.toml').write_text('[project]\nname = "demo"\n[tool.nbdev]\nnbs_path = "docs/notebooks"\ndoc_path = "public"\n')
(nbs/'_quarto.yml').write_text('website:\n  title: Demo\nformat:\n  html:\n    toc: false\n')
(nbs/'guide.qmd').write_text('# Guide')

7

A project with nested notebooks should still publish to its configured project-level output directory. The installed CLI must find that directory without a `--path` override.

Migration does not invent footer text. A project must supply its own footer.

In [ ]:
with working_directory(project): run('nbdev-migrate-octavo')
settings = octavo_config(nbs)
assert 'output' not in settings
test_eq(settings['toc_depth'], 0)
assert 'footer' not in settings
assert (nbs/'guide.md').exists() and (nbs/'_quarto.yml').exists()
with working_directory(project): run('nbdev-docs')
assert (project/'public/guide.html').exists()

test_eq(_octavo_output(nbs, {'project': {'output-dir': 'public'}}, {}), None)
test_eq(_octavo_output(nbs, {'project': {'output-dir': 'website'}}, {}), '../../website')
test_eq(_octavo_output(nbs, {'project': {'output-dir': 'website'}}, {'project': {'output-dir': 'public'}}), None)
test_eq(_octavo_output(nbs, {'project': {'output-dir': str(project/'elsewhere')}}, {}), '../../elsewhere')

An existing Octavo configuration belongs to the author. Repeating migration must preserve it rather than regenerate defaults.

In [ ]:
before = (nbs/'octavo.yml').read_text()
test_eq(_migrate_octavo(nbs), {})
test_eq((nbs/'octavo.yml').read_text(), before)
nested_tmp.cleanup()

Valid YAML is not always valid configuration. Lists, booleans, and numbers must fail before source changes, whether they came from Quarto or an existing Octavo configuration.

In [ ]:
with tempfile.TemporaryDirectory() as d:
    nbs = Path(d)
    (nbs/'guide.qmd').write_text('# Guide')
    (nbs/'_quarto.yml').write_text('website:\n  title: Demo\n')
    for invalid in ('[]', 'false', '0'):
        (nbs/'octavo.yml').write_text(invalid)
        with expect_fail(ValueError, contains='mapping'): _migrate_octavo(nbs)
        assert (nbs/'guide.qmd').exists() and (nbs/'_quarto.yml').exists()
    (nbs/'octavo.yml').unlink()
    for invalid in ('[]', 'false', '0'):
        (nbs/'_quarto.yml').write_text(invalid)
        with expect_fail(ValueError, contains='mapping'): _migrate_octavo(nbs)
        assert (nbs/'guide.qmd').exists() and not (nbs/'guide.md').exists()